# Manufacturing executor testbench

Interactive testbench for the `manufacturing_executor` lifecycle node and its `~/manufacture` action. Everything runs simulated / in local mode: Gazebo (robot with camera, tool rack and tool mount, conveyor, RFID scanner), MoveIt 2, the tools manager with two dispensing endtools and pattern executors, the `dispensing_optimizer`, the `mock_localizer`, the repositories and the production manager.

Prerequisites:
1. Build the workspace (`colcon build`) and use a Jupyter kernel from `/opt/venv` with ROS 2 Jazzy and the workspace overlay sourced.
2. Start the testbench in a terminal; it copies the local-mode files to `/tmp/station` and launches everything:
   ```bash
   src/manufacturing_executor/tests/testbench/setup.sh               # Gazebo GUI + RViz
   src/manufacturing_executor/tests/testbench/setup.sh gui:=false rviz:=false
   ```
3. Make sure the kernel uses the same `ROS_DOMAIN_ID` as the launch terminal (`setup.sh` defaults to `42`).

All lifecycle nodes start **Unconfigured**; the cells below bring them up. The tools manager configures the tool rack, the tool mount and the endtools itself, so those are not touched here. The manufacturing executor configures the localizer, the optimizer and the pattern executors when it is activated.

Scenario data (see `assets/mock_assembly_file.json` and `config/tools_manager_config.yaml`):
- The tool `abc123` (material `SXN123`) is mounted, the tool `def123` (material `SXN999`) is on rack slot 2.
- The operation needs the end tool `KRC123/R1A` with the material `SXN999/R1A`, so the loop has to unmount `abc123` and mount `def123`.
- The product `12345` / `1.0` / `SN-0001` arrives on the carrier; the mock localizer takes images and publishes `product_origin`.

Scenario: bring up the dependencies → activate the manufacturing executor → set the operation → spawn the product → send the `manufacture` goal → move the carrier in → watch the loop → cancel (or let the loop eject the product and wait for the next one).

The carrier is moved with `gz topic` (see **Carrier**); the Gazebo GUI Joint Position Controller (model `station`, joint `base_carrier_joint`: `-0.6` = in place, `0.0` = out) works as well. The simulation can run slowly, especially without a GPU; the first product spawn loads a large mesh and may need a second try.

## Setup

In [ ]:
import json
import re
import subprocess
import threading
import time
from pathlib import Path

import rclpy
from action_msgs.msg import GoalStatus
from lifecycle_msgs.msg import Transition
from lifecycle_msgs.srv import ChangeState, GetAvailableTransitions, GetState
from rclpy.action import ActionClient
from rclpy.executors import MultiThreadedExecutor
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy
from std_msgs.msg import Bool

from manufacturing_executor.action import Manufacture
from production_manager.srv import SetOperation
from station_workplace.msg import WorkplaceState
from station_workplace.srv import RemoveProduct, SpawnProduct
from tools_manager.msg import ToolInfo

TARGET_NODE = '/manufacturing_executor'

# Lifecycle nodes that must be Active before the manufacturing executor is activated, in bring-up order.
# Not listed: tool_rack / tool_mount / the endtools (configured by the tools manager) and the localizer,
# the optimizer and the pattern executors (configured by the manufacturing executor).
DEPENDENCY_NODES = [
    '/movement_controller', '/gazebo_world_manager', '/moveit2_world_manager', '/station_cache',
    '/object_model_storage', '/operation_repository', '/image_repository', '/production_manager',
    '/conveyor', '/simatic_rf240r_ascii', '/tools_manager', '/workplace_rfid_scanner',
]
# Released again (in this order) on bring-down; the first ones are owned by the manufacturing executor.
OWNED_NODES = ['/mock_localizer', '/dispensing_optimizer', '/dispensing_executor_abc123', '/dispensing_executor_def123']

# Must match testbench.launch.py (station_id) and assets/mock_assembly_file.json
STATION_ID = 'c330289c-0db3-4b5b-9fe0-3d9b63bedb72'
OPERATION_ID = 'dab06081-0e24-4583-9f33-06d3c26b69f7'
PRODUCT = {'product_number': '12345', 'product_revision': '1.0', 'serial_number': 'SN-0001'}
# SDF product model and its pose on the carrier [x, y, z, roll, pitch, yaw] relative to the carrier link;
# the in-place carrier puts the product at about (0.85, 0.25, 0.86) in the world.
# The product is turned by -60 deg about Z so that the optimizer camera poses (image-right
# follows the product X axis) are reachable by the arm.
ASSEMBLY_MODEL_ID = 'fd71afb0-d3e9-428c-a7e6-b2613b877088'
SPAWN_POSE = [-0.35, 0.25, 0.02, 0.0, 0.0, 0.0]  # yaw must match estimated_product_origin_pose in assets/local_storage.json
CARRIER_IN, CARRIER_OUT = -0.6, 0.0
IMAGES_DIR = Path('/tmp/station/images')

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('manufacturing_executor_testbench')
executor = MultiThreadedExecutor(num_threads=4)
executor.add_node(node)
spin_thread = threading.Thread(target=executor.spin, daemon=True)
spin_thread.start()

In [ ]:
def call(service_type, service_name, request, timeout=10.0):
    client = node.create_client(service_type, service_name)
    try:
        if not client.wait_for_service(timeout_sec=timeout):
            raise TimeoutError(f'Service {service_name} was not available within {timeout}s')
        future = client.call_async(request)
        deadline = time.monotonic() + timeout
        while not future.done():
            if time.monotonic() > deadline:
                raise TimeoutError(f'Service call {service_name} did not complete within {timeout}s')
            time.sleep(0.01)
        return future.result()
    finally:
        node.destroy_client(client)

def get_state(target=TARGET_NODE):
    return call(GetState, f'{target}/get_state', GetState.Request()).current_state.label

def available_transitions(target=TARGET_NODE):
    response = call(GetAvailableTransitions, f'{target}/get_available_transitions', GetAvailableTransitions.Request())
    return [t.transition.label for t in response.available_transitions]

def transition(transition_id, target=TARGET_NODE, timeout=60.0):
    request = ChangeState.Request()
    request.transition.id = transition_id
    success = call(ChangeState, f'{target}/change_state', request, timeout=timeout).success
    print(f'{target}: success={success}, state={get_state(target)}')
    return success

def bring_up(target):
    """Configure (if needed) and activate a lifecycle node; an active node is left as is."""
    if get_state(target) == 'unconfigured':
        transition(Transition.TRANSITION_CONFIGURE, target)
    if get_state(target) == 'inactive':
        transition(Transition.TRANSITION_ACTIVATE, target)
    print(f'{target}: {get_state(target)}')

def bring_down(target):
    """Deactivate and clean up a lifecycle node, whatever its current state."""
    if get_state(target) == 'active':
        transition(Transition.TRANSITION_DEACTIVATE, target)
    if get_state(target) == 'inactive':
        transition(Transition.TRANSITION_CLEANUP, target)
    print(f'{target}: {get_state(target)}')

## Monitor

Subscribes to the latched topics the loop reacts to; changes are printed as they arrive:
- `/workplace_rfid_scanner/workplace`: the product in the station (the loop fails if the product leaves unexpectedly),
- `/tool_mount/tool_mounted`: the mounted tool,
- `/mock_localizer/localized`: whether the localization is valid.

Inspect `latest` at any time.

In [ ]:
latest = {'workplace': None, 'tool_mounted': None, 'localized': None}

def _on_workplace(msg):
    latest['workplace'] = msg
    print(f'[workplace] in_place={msg.in_place} product_data={[p.serial_number for p in msg.product_data]}')

def _on_tool_mounted(msg):
    previous = latest['tool_mounted']
    latest['tool_mounted'] = msg
    if previous is None or previous.tool_sn != msg.tool_sn:
        print(f'[tool_mount] mounted tool: {msg.tool_sn or "(none)"}')

def _on_localized(msg):
    latest['localized'] = msg.data
    print(f'[localizer] localized={msg.data}')

latched_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE, durability=DurabilityPolicy.TRANSIENT_LOCAL)
subscriptions = [
    node.create_subscription(WorkplaceState, '/workplace_rfid_scanner/workplace', _on_workplace, latched_qos),
    node.create_subscription(ToolInfo, '/tool_mount/tool_mounted', _on_tool_mounted, latched_qos),
    node.create_subscription(Bool, '/mock_localizer/localized', _on_localized, latched_qos),
]

In [ ]:
latest

## Dependencies: bring up

Configures and activates the nodes the manufacturing executor needs to be `active`: movement controller, world managers (the workplace and the tools manager only activate while they are active), station cache (TCP calibration of the endtools), repositories, production manager, conveyor, RFID scanner, tools manager and the station workplace. Safe to re-run: active nodes are skipped.

The tools manager takes some time: it spawns the tools in Gazebo and configures the tool rack, the tool mount and the endtools.

In [ ]:
for dependency in DEPENDENCY_NODES:
    bring_up(dependency)

## Manufacturing executor: lifecycle

`~/manufacture` exists only while the node is **Active**.

- **configure** reads the parameters (change them before with `ros2 param set /manufacturing_executor <name> <value>`, e.g. `image_capture_settling_s`).
- **activate** requires all dependencies to be `active` and the localizer, the optimizer and both pattern executors to be `unconfigured`; it configures and activates the localizer and configures the optimizer and the executors. If it fails, the node log names the node and its state.

In [ ]:
print(get_state(), available_transitions())

In [ ]:
transition(Transition.TRANSITION_CONFIGURE)

In [ ]:
transition(Transition.TRANSITION_ACTIVATE)

## Production manager: set operation

On injection the workplace node calls `get_production_data`, which only answers for the operation and serial number set here (and the production manager's `station_id`). `index` is the position of the operation in the product's operation sequence; the first one is `0`. After the loop reports the last operation of a product, the queue is cleared, so set the operation again before the next product.

In [ ]:
request = SetOperation.Request(
    operation_id=OPERATION_ID, serial_number=PRODUCT['serial_number'], station_id=STATION_ID, index=0)
response = call(SetOperation, '/production_manager/set_operation', request)
print(f'success={response.success} error_message={response.error_message!r}')

## Carrier

The conveyor carrier is moved by publishing the position of the joint `base_carrier_joint` of the Gazebo model `station` (`-0.6` = in place, `0.0` = out). `inject_product` / `eject_product` of the simulated conveyor are no-ops, so moving the carrier is what makes the workplace see the product arrive or leave. The robot must not be above the conveyor while the product arrives: the testbench scene starts the arm turned away from it.

In [ ]:
def move_carrier(position):
    """Command the carrier joint position [m]; the carrier needs a moment to get there."""
    subprocess.run(
        ['gz', 'topic', '-t', '/model/station/joint/base_carrier_joint/0/cmd_pos',
         '-m', 'gz.msgs.Double', '-p', f'data: {position}'],
        check=True, timeout=10.0)

def carrier_position():
    """Current position of the carrier joint [m]."""
    output = subprocess.run(
        ['gz', 'topic', '-e', '-t', '/conveyor/joint_state', '-n', '1'],
        capture_output=True, text=True, check=True, timeout=20.0).stdout
    return float(re.findall(r'position: (\S+)', output)[-1])

In [ ]:
# carrier_position()

## Simulation: spawn product (workplace node must be Active)

Spawns the carrier (pallet, with `PRODUCT` written to its RFID tag) at the current pose of the conveyor `carrier` link and the product on top of it; both are welded on spawn. Spawn while the carrier is **out** (`0.0`). Only one product can be spawned at a time; remove it first.

The spawn loads large meshes and the simulation is slow, so the workplace waits up to `simulation.world_manager_timeout_sec` (15 s) for the world manager. A failed spawn is rolled back; run **remove product** before retrying if models are left over.

In [ ]:
request = SpawnProduct.Request(**PRODUCT, assembly_model_id=ASSEMBLY_MODEL_ID, pose=SPAWN_POSE)
response = call(SpawnProduct, '/workplace_rfid_scanner/spawn_product', request, timeout=60.0)
print(f'success={response.success} error_message={response.error_message!r}')

## manufacture action (node must be Active)

`start_manufacture()` sends the goal and returns its goal handle; the result is printed when it arrives (status, `success`, `error_message`, `total_manufacturing_time_s`). The goal has no feedback and ends only when it is cancelled (`success=True`), a system failure occurs (`success=False`) or the node is deactivated (`success=False`). Only one goal runs at a time; a second one is rejected.

The loop then goes through: injection (waits for the carrier) → production data → operation → tool lookup → localization (the mock localizer moves the robot and takes images) → unmount `abc123` / mount `def123` → optimization → image capture, pattern execution, image capture per pattern group → `report_production` → ejection (waits for the carrier to leave).

Progress is visible in the launch terminal. The images the loop uploads end up in `/tmp/station/images` (see **Images**).

In [ ]:
STATUS = {GoalStatus.STATUS_SUCCEEDED: 'SUCCEEDED', GoalStatus.STATUS_ABORTED: 'ABORTED',
          GoalStatus.STATUS_CANCELED: 'CANCELED'}
manufacture_client = ActionClient(node, Manufacture, f'{TARGET_NODE}/manufacture')

def _print_result(future):
    response = future.result()
    result = response.result
    print(f'[manufacture] status={STATUS.get(response.status, response.status)} success={result.success} '
          f'error_message={result.error_message!r} total_manufacturing_time_s={result.total_manufacturing_time_s:.1f}')

def start_manufacture(timeout=5.0):
    if not manufacture_client.wait_for_server(timeout_sec=timeout):
        raise TimeoutError(f'{TARGET_NODE}/manufacture not available (is the node active?)')
    future = manufacture_client.send_goal_async(Manufacture.Goal())
    deadline = time.monotonic() + timeout
    while not future.done():
        if time.monotonic() > deadline:
            raise TimeoutError('Goal response not received')
        time.sleep(0.01)
    goal_handle = future.result()
    print(f'[manufacture] accepted={goal_handle.accepted}')
    if goal_handle.accepted:
        goal_handle.get_result_async().add_done_callback(_print_result)
    return goal_handle

In [ ]:
goal = start_manufacture()

### Move the carrier in

The loop waits for the product (`injection`). Move the carrier in (wait until `carrier_position()` is about `-0.6`, the simulation may be slow); the workplace then reads the RFID tag, estimates the product origin and the loop continues. While the product is in the station, the loop watches that it stays: moving the carrier out during the run is a system failure.

In [ ]:
move_carrier(CARRIER_IN)

### Status

Lifecycle state of the nodes the loop drives (the optimizer stays active after use, the pattern executor is only active while its operation runs), the mounted tool and the product in the station.

In [ ]:
for name in [TARGET_NODE, *OWNED_NODES, '/dispenser_abc123', '/dispenser_def123']:
    print(f'{name}: {get_state(name)}')
print('carrier position:', round(carrier_position(), 3))
print('mounted tool:', latest['tool_mounted'].tool_sn if latest['tool_mounted'] else None)
workplace = latest['workplace']
print('workplace:', None if workplace is None else (workplace.in_place, [p.serial_number for p in workplace.product_data]))

### Images

The loop uploads the images it takes (before and after the patterns of each group) to the image repository; `metadata` holds the operation id, the serial number and the capture pose `[x, y, z, roll, pitch, yaw]`.

In [ ]:
images = sorted(IMAGES_DIR.glob('*'), key=lambda path: path.stat().st_mtime)
print(f'{len(images)} files in {IMAGES_DIR}')
for path in images[-6:]:
    print(path.name, path.stat().st_size, 'bytes')
metadata_files = [path for path in images if path.suffix == '.json']
if metadata_files:
    print(json.dumps(json.loads(metadata_files[-1].read_text()), indent=2)[:600])

### After the operation: ejection

When the loop has reported the operation and `finished_in_station` is `true`, it waits for the product to leave (`ejection`). Move the carrier out; the loop then waits for the next product (set the operation again, spawn a new product, move the carrier in).

### Cancel

Cancelling ends the goal with `success=True`: running child goals are cancelled, the movement session is released and the pattern executor is deactivated. The optimizer stays active and the product stays in the station.

In [ ]:
move_carrier(CARRIER_OUT)

In [ ]:
# Cancel the last goal (success=True in the result)
goal.cancel_goal_async()

In [ ]:
# A second goal while one is running is rejected
second_goal = start_manufacture()

## Failure scenarios

Each of these ends the goal with `success=False` and an `error_message` (or, for an operation failure, reports the failure and ejects the product):

- **Product leaves unexpectedly**: start a goal, move the carrier in, and while the loop is running move the carrier out again. All running child goals are cancelled.
- **Deactivate during a goal**: the result reads `Manufacturing aborted: node deactivated`; the localizer, the optimizer and the pattern executors are released (see the lifecycle cells below).
- **Operation failure (tool or material missing)**: change the required tool or material in `assets/mock_assembly_file.json` (e.g. `end_tool.product_number`), copy it to `/tmp/station`, cycle `operation_repository` and `production_manager` (deactivate, cleanup, configure, activate), and run the scenario again. The loop reports `Incorrect routing, station does not have the required endtool or the material`, ejects the product and continues with the next one; the goal keeps running.

## Simulation: remove product

Detaches and removes the spawned product and carrier models. Move the carrier out (and let the loop finish the ejection) first.

In [ ]:
response = call(RemoveProduct, '/workplace_rfid_scanner/remove_product', RemoveProduct.Request(), timeout=60.0)
print(f'success={response.success} error_message={response.error_message!r}')

## Lifecycle: bring down

Deactivating the manufacturing executor while a goal is running aborts the goal. It also deactivates and cleans up the localizer, the optimizer and the pattern executors. After that the node can be activated again.

In [ ]:
transition(Transition.TRANSITION_DEACTIVATE)
print({name: get_state(name) for name in OWNED_NODES})

In [ ]:
transition(Transition.TRANSITION_CLEANUP)

In [ ]:
# Optional: bring the dependencies down as well (the tools manager releases the rack, the mount and the endtools)
for dependency in reversed(DEPENDENCY_NODES):
    bring_down(dependency)

## Teardown

In [ ]:
manufacture_client.destroy()
for subscription in subscriptions:
    node.destroy_subscription(subscription)
executor.shutdown()
spin_thread.join(timeout=5.0)
node.destroy_node()
rclpy.shutdown()